In [0]:
%run "../libs/libs"

In [0]:
%run "../common/define_path"

In [0]:
otracking_path

In [0]:
def read_bronze_paymentdetail():
    print('Reading the Bronze Table Data : ',end='')
    df_bronzepaymentdetail = (spark.readStream
                    .table(f"otracking.bronze.paymentdetail")
                    )
    print(f'Reading otracking.bronze.paymentdetail Success!')
    return df_bronzepaymentdetail

In [0]:
df_paymentdetail = read_bronze_paymentdetail()  # Ensure the function df_bronzepaymentdetail is defined or imported before use.

In [0]:
from pyspark.sql.functions import col, to_date
df_paymentdetail = df_paymentdetail.select(col("id"),
                               col("srno").alias("serialno"),
                               col("paymentdate"),
                               col("amount").alias("paidamount"),
                               col("cheque").alias("chequeno"),
                               col("refno"),
                               col("payingparty"),
                               col("receivingparty"),
                               col("bankname"),
                               col("details"))

                               

In [0]:
from pyspark.sql.functions import current_timestamp
df_paymentdetail = df_paymentdetail.withColumn("ingestion_time",current_timestamp())
df_paymentdetail = convert_to_float(df_paymentdetail, "paidamount")
df_paymentdetail = convert_to_int(df_paymentdetail, "id")
df_paymentdetail = convert_to_timestamp(df_paymentdetail, "paymentdate")

In [0]:
def write_silver_paymentdetail(StreamingDF):
      write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',f'{otracking_path}/silver/checkpoint/paymentdetail/')
                .outputMode('append')
                .queryName("silverpaymentdetailtream")
                .trigger(availableNow=True)
                .toTable("otracking.silver.paymentdetail")
            )
      write_StreamSilver.awaitTermination()

In [0]:
write_silver_paymentdetail(df_paymentdetail)